# ProteomeScout Update Pipeline

Thin orchestration notebook: each cell shells out to a script in `../pipeline/`. See `../README.md` for the full step-by-step description and standalone CLI usage.

## Manual steps before running this notebook
1. Copy the current released ProteomeScout dataset into `../Data/current/<species>/data.tsv` for every species being updated.
2. Copy `../config.example.json` to `../config.local.json` and set `data_root` to where external resource downloads (e.g. UniProt proteomes) live.
3. Add a row to `citations.tsv` for each new resource being integrated this cycle (see the "Register a resource" cell below) and record the resulting resource ID(s) in the `RESOURCE_IDS` cell.

PSP integration is currently inactive (PhosphoSitePlus suspended academic access as of September 2026); its cells are included but skipped by default.

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path
from datetime import datetime

import pandas as pd

REPO_ROOT = Path("..").resolve()
PIPELINE_DIR = REPO_ROOT / "pipeline"
DATA_CURRENT = REPO_ROOT / "Data" / "current"
DATA_NEW = REPO_ROOT / "Data" / "new"
sys.path.insert(0, str(PIPELINE_DIR))

config_path = REPO_ROOT / "config.local.json"
if not config_path.exists():
    raise FileNotFoundError(
        f"Missing {config_path}. Copy config.example.json to config.local.json and set data_root."
    )
with config_path.open() as f:
    data_root = Path(json.load(f)["data_root"]).expanduser()

uniprot_data_dir = data_root / "Uniprot_Proteome"


def run(cmd):
    """Run a pipeline CLI command (list of str args) and raise on failure."""
    print(f"$ {' '.join(cmd)}")
    result = subprocess.run(cmd, cwd=PIPELINE_DIR)
    if result.returncode != 0:
        raise RuntimeError(f"Command failed ({result.returncode}): {' '.join(cmd)}")

## Species configuration for this update cycle

In [ ]:
from species_config import load_species_config

# species treated as a full reference proteome (adds all SwissProt-NR records, not just those with PTMs)
SPECIES_IS_FULL_REFERENCE = {
    "human": True,
    "mouse": True,
    "rat": False,
    "cow": False,
    "fly": False,
    "yeast": False,
}

# every species run in this cycle must have static facts registered in species_config.json
unknown_species = set(SPECIES_IS_FULL_REFERENCE) - set(load_species_config())
if unknown_species:
    raise ValueError(f"Add these species to species_config.json before running: {unknown_species}")

# resource IDs for this cycle, from the citations.tsv registration step below
RESOURCE_IDS = {
    "uniprot": None,  # fill in after registering the resource
    # "psp": None,     # uncomment once PSP access resumes
}

## Step 0: Split the combined dataset by species (skip if already staged per-species)

In [ ]:
combined_file = DATA_CURRENT / "data.tsv"
if combined_file.exists():
    run([
        "python", "split_species.py",
        "--input-file", str(combined_file),
        "--output-dir", str(DATA_CURRENT),
    ])
else:
    print(f"{combined_file} not found; assuming data is already staged per-species under {DATA_CURRENT}")

## Step 1: Register a new resource ID (run once per resource, per cycle)

In [ ]:
import translationTools

citations_file = DATA_CURRENT / "ProteomeScout_Dataset" / "citations.tsv"
citations_file_new = DATA_NEW / "ProteomeScout_Dataset" / "citations.tsv"
citations_file_new.parent.mkdir(parents=True, exist_ok=True)

resource_id, citations_df = translationTools.return_new_resource_id(
    citations_file,
    Name=f"UniProt Proteome Integration {datetime.now().strftime('%Y-%m-%d')}",
    Citation="UniProt Consortium. UniProt: the universal protein knowledgebase in 2021. Nucleic Acids Res. 2021 Jan 8;49(D1):D480-D489. doi: 10.1093/nar/gkaa1100. PMID: 33237286; PMCID: PMC7779014.",
    Description="Integration of UniProt proteome data into ProteomeScout for this update cycle.",
    URL="https://www.uniprot.org/",
    PMID="33237286",
)
citations_df.to_csv(citations_file_new, sep="\t", index=False)
print(f"New UniProt resource ID: {resource_id}")
RESOURCE_IDS["uniprot"] = resource_id

## Step 2: Update all records to current UniProt

In [ ]:
for species in SPECIES_IS_FULL_REFERENCE:
    run([
        "python", "update_proteomescout.py",
        "--data-dir", str(DATA_CURRENT),
        "--species", species,
        "--batch-size", "500",
    ])
    updated_file = DATA_CURRENT / species / "data.tsv.updated_clean"
    df = pd.read_csv(updated_file, sep="\t")
    qc_issues = translationTools.run_quality_control_mods(df)
    print(f"{species}: {len(qc_issues)} QC issues after update step")

## Step 3: Integrate UniProt proteome data

In [ ]:
for species, is_full_reference in SPECIES_IS_FULL_REFERENCE.items():
    run([
        "python", "uniprot_integration_pipeline.py",
        "--species", species,
        "--uniprot-data-dir", str(uniprot_data_dir),
        "--pscout-data-dir", str(DATA_CURRENT),
        "--resource-id", str(RESOURCE_IDS["uniprot"]),
        "--keep-records-not-in-pscout", str(is_full_reference),
        "--uniprot-swiss-nr", str(is_full_reference),
    ])
    integrated_file = DATA_CURRENT / species / "data.tsv.uniprot"
    df = pd.read_csv(integrated_file, sep="\t")
    qc_issues = translationTools.run_quality_control_mods(df)
    print(f"{species}: {len(qc_issues)} QC issues after UniProt integration")

## Step 4 (inactive): Integrate PhosphoSitePlus

Skip until academic PSP access resumes. Left here so the cycle is easy to re-enable.

In [ ]:
RUN_PSP = False
if RUN_PSP:
    psp_data_dir = data_root / "PhosphoSitePlus"
    for species in SPECIES_IS_FULL_REFERENCE:
        run([
            "python", "psp_integration_pipeline.py",
            "--species", species,
            "--psp-data-file", str(psp_data_dir / "phosphositeplus_data.csv"),
            "--pscout-data-dir", str(DATA_CURRENT),
            "--resource-id", str(RESOURCE_IDS["psp"]),
            "--keep-records-not-in-pscout", "False",
            "--uniprot-swiss-nr", "False",
        ])

## Step 5: Reintegrate species files into one dataset

In [ ]:
# final per-species file is data.tsv.psp if PSP ran, otherwise data.tsv.uniprot
final_suffix = "data.tsv.psp" if RUN_PSP else "data.tsv.uniprot"

df_dict = {}
for species in SPECIES_IS_FULL_REFERENCE:
    species_file = DATA_CURRENT / species / final_suffix
    df_dict[species] = pd.read_csv(species_file, sep="\t")

combined_df = pd.concat(df_dict.values(), ignore_index=True)
DATA_NEW.mkdir(parents=True, exist_ok=True)
combined_output_file = DATA_NEW / "data.tsv"
combined_df.to_csv(combined_output_file, sep="\t", index=False)
print(f"Wrote combined dataset ({len(combined_df)} records) to {combined_output_file}")

## Step 6: Final QC

In [ ]:
run([
    "python", "qc_integrated_data.py",
    "--input-file", str(DATA_NEW / "data.tsv"),
    "--report-json", str(DATA_NEW / "qc_report.json"),
])